In [17]:
using Random
using LinearAlgebra
using JuMP
using SCS
using DynamicPolynomials
using SumOfSquares
using DataFrames
using Combinatorics


const MOI = JuMP.MOI
const DEFAULT_SOLVER = "SCS"
const DEFAULT_N = 10
const DEFAULT_SIGMA = 2.0

# using MosekTools
# const DEFAULT_SOLVER = "MOSEK"

2.0

Core helpers from `approx.jl`, with notebook defaults that run under `SCS`.


In [18]:
function optimizer_factory(solver_kind::AbstractString = DEFAULT_SOLVER)
    normalized = uppercase(strip(solver_kind))
    if normalized == "SCS"
        return optimizer_with_attributes(SCS.Optimizer)
    elseif normalized == "MOSEK"
        configure_mosek_license!()
        return optimizer_with_attributes(Mosek.Optimizer)
    end
end

function makeF_mle_trunc_closed(n::Int, x::AbstractVector, k::Int = n, sigma::Real = 1.0; seed = nothing)
    @assert length(x) == n "x must be length n"
    @assert 0 <= k <= n
    @assert sigma > 0
    seed === nothing || Random.seed!(seed)

    fqr = qr(randn(n, n))
    q = Matrix(fqr.Q)
    r = fqr.R
    q = q * Diagonal(sign.(diag(r)))
    eps_eig = 1e-6
    lambda_vals = eps_eig .+ (1 - eps_eig) .* rand(n)
    sigma_mat = q * Diagonal(lambda_vals) * q'
    sigma_mat = (sigma_mat + sigma_mat') / 2

    alpha = 1 / (sigma^2)
    f = zero(x[1]) + 1
    if k > 0
        for degree in 1:k
            for subset in combinations(1:n, degree)
                coeff = (alpha^degree) * det(@view sigma_mat[subset, subset])
                coeff == 0 && continue
                term = coeff
                for idx in subset
                    term *= x[idx]
                end
                f += term
            end
        end
    end
    return f, sigma_mat
end

function gamma_bruteforce(sigma_mat::AbstractMatrix, sigma::Real)
    a = sigma_mat / (sigma^2)
    n = size(a, 1)
    universe = collect(1:n)

    f = function (subset::Vector{Int})
        isempty(subset) && return 1.0
        return det(I + a[subset, subset])
    end

    gamma_val = Inf
    for lmask in 0:((Int(1) << n) - 1)
        lset = Int[]
        for idx in 1:n
            if ((lmask >> (idx - 1)) & 0x1) == 1
                push!(lset, idx)
            end
        end
        not_lset = setdiff(universe, lset)
        f_lset = f(lset)
        m = length(not_lset)
        m == 0 && continue

        for smask in 1:((Int(1) << m) - 1)
            sset = Int[]
            for idx in 1:m
                if ((smask >> (idx - 1)) & 0x1) == 1
                    push!(sset, not_lset[idx])
                end
            end

            denom = f(vcat(lset, sset)) - f_lset
            denom <= 0 && continue

            numer = 0.0
            for item in sset
                numer += f(vcat(lset, [item])) - f_lset
            end

            ratio = numer / denom
            if ratio < gamma_val
                gamma_val = ratio
            end
        end
    end

    return gamma_val
end

function gamma_spectral_ratio(sigma_mat::AbstractMatrix; sigma::Real)
    @assert sigma > 0
    a = I + (sigma_mat ./ (sigma^2))
    eigs = eigvals(Symmetric(Matrix(a)))
    sort!(eigs, rev = true)
    lambda_min = eigs[end]
    numer = size(sigma_mat, 1) * max(lambda_min - 1.0, 0.0)
    denom = try
        exp(logdet(Symmetric(Matrix(a)))) - 1.0
    catch
        det(Matrix(a)) - 1.0
    end
    return denom <= 0 ? 0.0 : (numer / denom)
end

function run_one(k::Int, t::Int, seed::Int; solver_kind::AbstractString = DEFAULT_SOLVER, n::Int = DEFAULT_N, sigma::Real = DEFAULT_SIGMA)
    @polyvar x[1:n] y[1:n]

    gamma_sos = NaN
    gamma_bf = NaN
    gamma_bf_time = NaN
    gamma_sr = NaN
    status_code = ""
    status_text = ""
    note = ""
    setup_time = NaN
    solve_time = NaN

    try
        f, sigma_mat = makeF_mle_trunc_closed(n, x, k, sigma; seed = seed)
        grad_f = [differentiate(f, xi) for xi in x]
        eps_val = det(I + sigma_mat ./ (sigma^2)) - f(x => ones(n))
        fx = f
        fy = f(x => y)

        model = SOSModel(optimizer_factory(solver_kind))
        set_silent(model)
        @variable(model, gamma)

        poly_constraints_x = [xi * (1 - xi) for xi in x]
        poly_constraints_y = [yi * (1 - yi) for yi in y]
        poly_constraints_xy = [x[idx] * y[idx] - x[idx] for idx in 1:n]
        domain = algebraicset(vcat(poly_constraints_x, poly_constraints_y, poly_constraints_xy))

        poly_sum = zero(fx)
        for idx in 1:n
            poly_sum += (y[idx] - x[idx]) * (grad_f[idx] - gamma * eps_val)
        end
        poly_sum += gamma * (fx - fy)

        @objective(model, Max, gamma)
        t_setup_start = time()
        @constraint(model, poly_sum >= 0, domain = domain, maxdegree = 2 * t)
        setup_time = time() - t_setup_start

        optimize!(model)
        solve_time = try
            MOI.get(backend(model), MOI.SolveTimeSec())
        catch
            NaN
        end

        term_status = termination_status(model)
        prim_status = primal_status(model)
        status_code = string(term_status)
        status_text = string(prim_status)

        if term_status in (MOI.OPTIMAL, MOI.LOCALLY_SOLVED)
            gamma_sos = value(gamma)
        else
            note = "Solver not optimal: $(status_code)"
        end

        t_bf_start = time()
        try
            gamma_bf = gamma_bruteforce(sigma_mat, sigma)
        catch err
            gamma_bf = NaN
            note == "" && (note = "gamma_bruteforce failed: $(sprint(showerror, err))")
        end
        gamma_bf_time = time() - t_bf_start

        try
            gamma_sr = gamma_spectral_ratio(sigma_mat; sigma = sigma)
        catch err
            gamma_sr = NaN
            note == "" && (note = "gamma_spectral_ratio failed: $(sprint(showerror, err))")
        end
    catch err
        msg = sprint(showerror, err)
        note == "" && (note = "Run failed: $(msg)")
    end

    return (
        n = n,
        t = t,
        k = k,
        seed = seed,
        solver = uppercase(strip(solver_kind)),
        time_source = "moi_solve_time_sec",
        gamma_SOS = gamma_sos,
        gamma_bruteforce = gamma_bf,
        gamma_bruteforce_time = gamma_bf_time,
        gamma_spectral_ratio = gamma_sr,
        status_code = status_code,
        status_text = status_text,
        note = note,
        setup_time = setup_time,
        solve_time = solve_time,
    )
end

function run_experiment(; solver_kind::AbstractString = DEFAULT_SOLVER, pair_specs = [(1, 1)], seeds = [0, 1], n::Int = DEFAULT_N, sigma::Real = DEFAULT_SIGMA)
    rows = NamedTuple[]
    for (k, t) in pair_specs
        for seed in seeds
            push!(rows, run_one(k, t, seed; solver_kind = solver_kind, n = n, sigma = sigma))
        end
    end
    return DataFrame(rows)
end


run_experiment (generic function with 1 method)

Small demo under `SCS`.

In [19]:
one_run = DataFrame([run_one(1, 1, 0; n = 4, sigma = 2.0, solver_kind = "SCS")])
one_run


Row,n,t,k,seed,solver,time_source,gamma_SOS,gamma_bruteforce,gamma_bruteforce_time,gamma_spectral_ratio,status_code,status_text,note,setup_time,solve_time
,Int64,Int64,Int64,Int64,String,String,Float64,Float64,Float64,Float64,String,String,String,Float64,Float64
1,4,1,1,0,SCS,moi_solve_time_sec,0.711629,0.939132,4.79221e-5,0.436239,OPTIMAL,FEASIBLE_POINT,,0.014266,0.002639


Demo mirroring experiment setup in 5.2

In [16]:
experiment_df = run_experiment(; pair_specs = [(2, 2)], seeds = [0, 1, 2], n = 10, sigma = 2.0, solver_kind = "SCS")
experiment_df


Row,n,t,k,seed,solver,time_source,gamma_SOS,gamma_bruteforce,gamma_bruteforce_time,gamma_spectral_ratio,status_code,status_text,note,setup_time,solve_time
,Int64,Int64,Int64,Int64,String,String,Float64,Float64,Float64,Float64,String,String,String,Float64,Float64
1,10,2,2,0,SCS,moi_solve_time_sec,0.38159,0.653726,0.0642071,0.113921,OPTIMAL,FEASIBLE_POINT,,0.00264788,2.35669
2,10,2,2,1,SCS,moi_solve_time_sec,0.21136,0.521912,0.0645602,0.103698,OPTIMAL,FEASIBLE_POINT,,0.00267005,3.69242
3,10,2,2,2,SCS,moi_solve_time_sec,0.5411,0.740015,0.0640121,0.122317,OPTIMAL,FEASIBLE_POINT,,0.00256801,6.30274
